# 使用 OpenAPI 规范进行函数调用


互联网的很大一部分由 RESTful API 驱动。赋予 GPT 调用这些 API 的能力会打开无限可能。本 notebook 演示如何使用 GPT 智能地调用 API。它利用了 OpenAPI 规范和链式函数调用。

[OpenAPI 规范（OAS）](https://swagger.io/specification/)是一种被普遍接受的标准，用于以机器可读、可解析的格式描述 RESTful API 的细节。它让人和计算机都能理解服务的能力，也可以用来向 GPT 展示如何调用 API。

本 notebook 分为两个主要部分：

1. 如何将示例 OpenAPI 规范转换为供 chat completions API 使用的函数定义列表。
2. 如何使用 chat completions API 根据用户指令智能地调用这些函数。

建议在继续之前先熟悉[函数调用](./How_to_call_functions_with_chat_models.ipynb)。


In [1]:
!pip install -q jsonref # for resolving $ref's in the OpenAPI spec
!pip install -q openai

DEPRECATION: textract 1.6.5 has a non-standard dependency specifier extract-msg<=0.29.*. pip 23.3 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of textract or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.2.1 -> 23.3.1
[notice] To update, run: pip install --upgrade pip
DEPRECATION: textract 1.6.5 has a non-standard dependency specifier extract-msg<=0.29.*. pip 23.3 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of textract or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.2.1 -> 23.3.1
[notice] To update, run: pip install --upgrade pip


In [7]:
import os
import json
import jsonref
from openai import OpenAI
import requests
from pprint import pp

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as env var>"))

## 如何将 OpenAPI 规范转换为函数定义


这里使用的示例 OpenAPI 规范是用 `gpt-4` 创建的。我们会将这个示例规范转换为一组可提供给 chat completion API 的函数定义。模型会根据提供的用户指令，生成包含调用这些函数所需参数的 JSON 对象。

在继续之前，我们先查看一下这份生成的规范。OpenAPI 规范包含 API 端点、它们支持的操作、接受的参数、能处理的请求以及返回的响应等详细信息。该规范以 JSON 格式定义。

规范中的端点包括以下操作：

- 列出所有事件
- 创建新事件
- 按 ID 获取事件
- 按 ID 删除事件
- 按 ID 更新事件名称

规范中的每个操作都有一个 `operationId`，在将规范解析为函数规格时，我们会将其用作函数名。规范中还包含定义每个操作参数的数据类型和结构的 schema。

你可以在这里查看 schema：


In [3]:
with open('./data/example_events_openapi.json', 'r') as f:
    openapi_spec = jsonref.loads(f.read()) # it's important to load with jsonref, as explained below

display(openapi_spec)

{'openapi': '3.0.0',
 'info': {'version': '1.0.0',
  'title': 'Event Management API',
  'description': 'An API for managing event data'},
 'paths': {'/events': {'get': {'summary': 'List all events',
    'operationId': 'listEvents',
    'responses': {'200': {'description': 'A list of events',
      'content': {'application/json': {'schema': {'type': 'array',
         'items': {'type': 'object',
          'properties': {'id': {'type': 'string'},
           'name': {'type': 'string'},
           'date': {'type': 'string', 'format': 'date-time'},
           'location': {'type': 'string'}},
          'required': ['name', 'date', 'location']}}}}}}},
   'post': {'summary': 'Create a new event',
    'operationId': 'createEvent',
    'requestBody': {'required': True,
     'content': {'application/json': {'schema': {'type': 'object',
        'properties': {'id': {'type': 'string'},
         'name': {'type': 'string'},
         'date': {'type': 'string', 'format': 'date-time'},
         'location

现在我们已经很好地理解了这份 OpenAPI 规范，可以着手将其解析为函数规格了。

我们可以编写一个简单的 `openapi_to_functions` 函数来生成定义列表，其中每个函数都表示为一个包含以下键的字典：

- `name`：对应 OpenAPI 规范中定义的 API 端点的操作标识符。
- `description`：函数的简要描述或摘要，概述函数的作用。
- `parameters`：定义函数预期输入参数的 schema。它提供了每个参数的类型、是必需还是可选，以及其他相关细节。

对于 schema 中定义的每个端点，我们需要执行以下操作：

1. **解析 JSON 引用**：在 OpenAPI 规范中，通常会使用 JSON 引用（也称为 $ref）来避免重复。这些引用指向在多处使用的定义。例如，如果多个 API 端点返回相同的对象结构，就可以只定义一次该结构，然后在需要的地方引用它。我们需要解析这些引用，并将其替换为所指向的内容。

2. **提取函数名：** 我们直接使用 operationId 作为函数名。另外，也可以使用端点路径和操作作为函数名。

3. **提取描述和参数：** 我们会遍历 `description`、`summary`、`requestBody` 和 `parameters` 字段，以填充函数的描述和参数。

实现如下：


In [9]:
def openapi_to_functions(openapi_spec):
    functions = []

    for path, methods in openapi_spec["paths"].items():
        for method, spec_with_ref in methods.items():
            # 1. Resolve JSON references.
            spec = jsonref.replace_refs(spec_with_ref)

            # 2. Extract a name for the functions.
            function_name = spec.get("operationId")

            # 3. Extract a description and parameters.
            desc = spec.get("description") or spec.get("summary", "")

            schema = {"type": "object", "properties": {}}

            req_body = (
                spec.get("requestBody", {})
                .get("content", {})
                .get("application/json", {})
                .get("schema")
            )
            if req_body:
                schema["properties"]["requestBody"] = req_body

            params = spec.get("parameters", [])
            if params:
                param_properties = {
                    param["name"]: param["schema"]
                    for param in params
                    if "schema" in param
                }
                schema["properties"]["parameters"] = {
                    "type": "object",
                    "properties": param_properties,
                }

            functions.append(
                {"type": "function", "function": {"name": function_name, "description": desc, "parameters": schema}}
            )

    return functions


functions = openapi_to_functions(openapi_spec)

for function in functions:
    pp(function)
    print()


{'type': 'function',
 'function': {'name': 'listEvents',
              'description': 'List all events',
              'parameters': {'type': 'object', 'properties': {}}}}

{'type': 'function',
 'function': {'name': 'createEvent',
              'description': 'Create a new event',
              'parameters': {'type': 'object',
                             'properties': {'requestBody': {'type': 'object',
                                                            'properties': {'id': {'type': 'string'},
                                                                           'name': {'type': 'string'},
                                                                           'date': {'type': 'string',
                                                                                    'format': 'date-time'},
                                                                           'location': {'type': 'string'}},
                                                            'required':

## 如何使用 GPT 调用这些函数


有了这些函数定义后，我们就可以借助 GPT 根据用户输入智能地调用它们。

需要注意的是，chat completions API 并不会执行函数；它只会生成 JSON，供你在自己的代码中用来调用函数。

关于函数调用的更多信息，请参阅我们专门的[函数调用指南](./How_to_call_functions_with_chat_models.ipynb)。


In [23]:
SYSTEM_MESSAGE = """
You are a helpful assistant.
Respond to the following prompt by using function_call and then summarize actions.
Ask for clarification if a user request is ambiguous.
"""

# Maximum number of function calls allowed to prevent infinite or lengthy loops
MAX_CALLS = 5


def get_openai_response(functions, messages):
    return client.chat.completions.create(
        model="gpt-3.5-turbo-16k",
        tools=functions,
        tool_choice="auto",  # "auto" means the model can pick between generating a message or calling a function.
        temperature=0,
        messages=messages,
    )


def process_user_instruction(functions, instruction):
    num_calls = 0
    messages = [
        {"content": SYSTEM_MESSAGE, "role": "system"},
        {"content": instruction, "role": "user"},
    ]

    while num_calls < MAX_CALLS:
        response = get_openai_response(functions, messages)
        message = response.choices[0].message
        print(message)
        try:
            print(f"\n>> Function call #: {num_calls + 1}\n")
            pp(message.tool_calls)
            messages.append(message)

            # For the sake of this example, we'll simply add a message to simulate success.
            # Normally, you'd want to call the function here, and append the results to messages.
            messages.append(
                {
                    "role": "tool",
                    "content": "success",
                    "tool_call_id": message.tool_calls[0].id,
                }
            )

            num_calls += 1
        except:
            print("\n>> Message:\n")
            print(message.content)
            break

    if num_calls >= MAX_CALLS:
        print(f"Reached max chained function calls: {MAX_CALLS}")


USER_INSTRUCTION = """
Instruction: Get all the events.
Then create a new event named AGI Party.
Then delete event with id 2456.
"""

process_user_instruction(functions, USER_INSTRUCTION)


ChatCompletionMessage(content=None, role='assistant', function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_jmlvEyMRMvOtB80adX9RbqIV', function=Function(arguments='{}', name='listEvents'), type='function')])

>> Function call #: 1

[ChatCompletionMessageToolCall(id='call_jmlvEyMRMvOtB80adX9RbqIV', function=Function(arguments='{}', name='listEvents'), type='function')]
ChatCompletionMessage(content=None, role='assistant', function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_OOPOY7IHMq3T7Ib71JozlUQJ', function=Function(arguments='{\n  "requestBody": {\n    "id": "1234",\n    "name": "AGI Party",\n    "date": "2022-12-31",\n    "location": "New York"\n  }\n}', name='createEvent'), type='function')])

>> Function call #: 2

[ChatCompletionMessageToolCall(id='call_OOPOY7IHMq3T7Ib71JozlUQJ', function=Function(arguments='{\n  "requestBody": {\n    "id": "1234",\n    "name": "AGI Party",\n    "date": "2022-12-31",\n    "location": "New York"\n  }\n}', name='c

### 结论

我们演示了如何将 OpenAPI 规范转换为可交给 GPT 的函数规格，使其能够智能地调用这些函数，并展示了如何将这些调用串联起来执行复杂操作。

该系统可能的扩展包括：处理需要条件逻辑或循环的更复杂用户指令、与真实 API 集成以执行实际操作，以及改进错误处理和验证，以确保指令可行且函数调用成功。